# 03 · Feature Engineering — Indicadores Técnicos y Lags
**Diplomado en Ciencia de Datos · ENES UNAM León**

Construimos las **variables de entrada (features)** para nuestros modelos. El principio es darle al modelo información del pasado para que pueda predecir el precio del día siguiente.

### Features que vamos a crear:
| Categoría | Feature | Descripción |
|---|---|---|
| **Lags** | Close_lag_1..7 | Precio de cierre de los últimos N días |
| **Tendencia** | SMA_7, SMA_21 | Media móvil simple (7 y 21 días) |
| **Tendencia** | EMA_12, EMA_26 | Media móvil exponencial |
| **Momentum** | RSI_14 | Relative Strength Index |
| **Momentum** | MACD, MACD_signal | Moving Average Convergence Divergence |
| **Volatilidad** | BB_upper, BB_lower | Bandas de Bollinger |
| **Volatilidad** | volatility_7d | Desv. estándar de retornos 7 días |
| **Volumen** | Volume_change | Cambio % en volumen |
| **Calendario** | day_of_week | Día de la semana (0=lunes) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

df = pd.read_csv('../data/processed/sol_usd_eda.csv', index_col=0, parse_dates=True)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Datos cargados: {df.shape}')
df.head(3)

## 3.1 Lags del precio de cierre

In [ ]:
# Lags: precio de cierre de los últimos N días
# Esto le dice al modelo: "el precio de hoy depende de los precios anteriores"
for lag in [1, 2, 3, 5, 7, 14]:
    df[f'Close_lag_{lag}'] = df['Close'].shift(lag)

print('Lags creados ✓')
df[['Close', 'Close_lag_1', 'Close_lag_2', 'Close_lag_7']].tail(5)

## 3.2 Medias Móviles (SMA y EMA)

In [ ]:
# SMA — Simple Moving Average
df['SMA_7']  = df['Close'].rolling(window=7).mean()
df['SMA_21'] = df['Close'].rolling(window=21).mean()

# EMA — Exponential Moving Average (da más peso a datos recientes)
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()

# Visualizar
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], label='Close', color='gray', alpha=0.6, linewidth=1)
plt.plot(df.index, df['SMA_7'],  label='SMA 7',  color='#9945FF', linewidth=1.5)
plt.plot(df.index, df['SMA_21'], label='SMA 21', color='#14F195', linewidth=1.5)
plt.plot(df.index, df['EMA_12'], label='EMA 12', color='#FFB800', linewidth=1.5, linestyle='--')
plt.title('SOL-USD con Medias Móviles')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/03_medias_moviles.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 RSI — Relative Strength Index

Mide velocidad y magnitud de los cambios de precio. Rango 0–100:
- **> 70** → sobrecomprado (posible corrección)
- **< 30** → sobrevendido (posible rebote)

In [ ]:
def compute_rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=window - 1, min_periods=window).mean()
    avg_loss = loss.ewm(com=window - 1, min_periods=window).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['RSI_14'] = compute_rsi(df['Close'], 14)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df['Close'], color='#9945FF', linewidth=1.5)
axes[0].set_title('Precio de Cierre')
axes[0].set_ylabel('USD')

axes[1].plot(df.index, df['RSI_14'], color='#FFB800', linewidth=1.5)
axes[1].axhline(70, color='red', linestyle='--', alpha=0.7, label='Sobrecomprado (70)')
axes[1].axhline(30, color='green', linestyle='--', alpha=0.7, label='Sobrevendido (30)')
axes[1].fill_between(df.index, 70, df['RSI_14'].clip(lower=70), alpha=0.2, color='red')
axes[1].fill_between(df.index, df['RSI_14'].clip(upper=30), 30, alpha=0.2, color='green')
axes[1].set_title('RSI (14 días)')
axes[1].set_ylim(0, 100)
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/raw/03_rsi.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 MACD

In [ ]:
# MACD = EMA_12 - EMA_26
df['MACD']        = df['EMA_12'] - df['EMA_26']
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_hist']   = df['MACD'] - df['MACD_signal']

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df['Close'], color='#9945FF', linewidth=1.5)
axes[0].set_title('Precio de Cierre')
axes[0].set_ylabel('USD')

axes[1].plot(df.index, df['MACD'], label='MACD', color='#14F195', linewidth=1.5)
axes[1].plot(df.index, df['MACD_signal'], label='Signal', color='#FF4444', linewidth=1.5)
colors_macd = ['#14F195' if v >= 0 else '#FF4444' for v in df['MACD_hist'].fillna(0)]
axes[1].bar(df.index, df['MACD_hist'], color=colors_macd, alpha=0.5, label='Histograma')
axes[1].axhline(0, color='white', linewidth=0.5)
axes[1].set_title('MACD')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/raw/03_macd.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Bandas de Bollinger

In [ ]:
window_bb = 20
df['BB_mid']   = df['Close'].rolling(window_bb).mean()
df['BB_std']   = df['Close'].rolling(window_bb).std()
df['BB_upper'] = df['BB_mid'] + 2 * df['BB_std']
df['BB_lower'] = df['BB_mid'] - 2 * df['BB_std']
df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / df['BB_mid']  # feature de volatilidad

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], label='Close', color='#9945FF', linewidth=1.5)
plt.plot(df.index, df['BB_upper'], label='Banda superior', color='red', linewidth=1, linestyle='--')
plt.plot(df.index, df['BB_mid'], label='SMA 20', color='white', linewidth=1, linestyle=':')
plt.plot(df.index, df['BB_lower'], label='Banda inferior', color='green', linewidth=1, linestyle='--')
plt.fill_between(df.index, df['BB_lower'], df['BB_upper'], alpha=0.05, color='gray')
plt.title('Bandas de Bollinger (20 días, ±2σ)')
plt.ylabel('USD')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/03_bollinger.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Features de volumen y calendario

In [ ]:
# Cambio porcentual en volumen
df['Volume_change'] = df['Volume'].pct_change() * 100
df['Volume_SMA7']   = df['Volume'].rolling(7).mean()

# Features de calendario
df['day_of_week'] = df.index.dayofweek  # 0=lunes, 6=domingo
df['month']       = df.index.month

print('Features adicionales creados ✓')

## 3.7 Variable objetivo: precio del día siguiente

In [ ]:
# Target: precio de cierre del día SIGUIENTE
# shift(-1) mueve el precio un día hacia atrás (futuro)
df['Target'] = df['Close'].shift(-1)

print('Variable objetivo (Target = Close del día siguiente) creada ✓')
df[['Close', 'Target']].tail(5)

## 3.8 Limpieza final — eliminar NaN generados por rolling windows

In [ ]:
print(f'Filas antes de limpiar : {len(df)}')
print(f'NaN por columna:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

df.dropna(inplace=True)

print(f'\nFilas después de limpiar : {len(df)}')
print('Todas las columnas sin NaN ✓')

## 3.9 Resumen de features y correlación con el Target

In [ ]:
feature_cols = [
    'Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_7', 'Close_lag_14',
    'SMA_7', 'SMA_21', 'EMA_12', 'EMA_26',
    'RSI_14', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width',
    'volatility_21d', 'Volume_change', 'Volume_SMA7',
    'day_of_week', 'month'
]

corr_target = df[feature_cols + ['Target']].corr()['Target'].drop('Target').sort_values(ascending=False)

plt.figure(figsize=(10, 7))
colors = ['#14F195' if v > 0 else '#FF4444' for v in corr_target]
corr_target.plot(kind='barh', color=colors, edgecolor='none')
plt.axvline(0, color='white', linewidth=0.8)
plt.title('Correlación de features con el Target (precio del día siguiente)')
plt.xlabel('Correlación de Pearson')
plt.tight_layout()
plt.savefig('../data/raw/03_correlacion_target.png', dpi=150, bbox_inches='tight')
plt.show()

print(corr_target)

## 3.10 Guardar dataset final

In [ ]:
df.to_csv('../data/processed/sol_usd_features.csv')
print(f'Dataset con features guardado ✓')
print(f'Shape final : {df.shape}')
print(f'Features    : {len(feature_cols)}')
print(f'Período     : {df.index.min().date()} → {df.index.max().date()}')

---
## ✅ Features creados

```python
FEATURE_COLS = [
    'Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Close_lag_5', 'Close_lag_7', 'Close_lag_14',
    'SMA_7', 'SMA_21', 'EMA_12', 'EMA_26',
    'RSI_14', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width',
    'volatility_21d', 'Volume_change', 'Volume_SMA7',
    'day_of_week', 'month'
]
TARGET = 'Target'  # Close del día siguiente
```

**Siguiente paso →** `04_modeling.ipynb`